# KnowGen Multi-Agent System Test
## Testing Supervisor Agent & Retriever Agent with CNXHKH.docx

This notebook tests the two core agents on Vietnamese academic document (CNXHKH.docx)
- **Supervisor Agent**: Task classification, language detection, execution planning
- **Retriever Agent**: Query rewriting, document retrieval, ranking, evidence extraction

## Section 1: Setup Environment & Import Libraries

In [2]:
import sys
import os
import json
import logging
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
from dotenv import load_dotenv

# Setup paths
backend_dir = Path.cwd().parent.parent if "agents" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(backend_dir))

# Load environment variables
load_dotenv()

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration
CNXHKH_FILE = r"H:\ĐH 1,2\BT CNXHKH.docx" # Vietnamese document path
VECTOR_STORE_DIR = Path(os.getcwd()) / "vector_store" / "faiss_index"

print(f"✓ Backend dir: {backend_dir}")
print(f"✓ Document file: {CNXHKH_FILE}")
print(f"✓ Vector store dir: {VECTOR_STORE_DIR}")
print(f"✓ Ready to test agents!")

✓ Backend dir: h:\KnowGen Project\backend
✓ Document file: H:\ĐH 1,2\BT CNXHKH.docx
✓ Vector store dir: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
✓ Ready to test agents!


## Section 2: Install & Verify Dependencies

In [3]:
import subprocess
import sys

# Define required packages
required_packages = [
    'langgraph',
    'langchain',
    'langchain-core',
    'langchain-openai',
    'langchain-huggingface',
    'langchain-community',
    'faiss-cpu',
    'python-docx',
    'pandas',
    'python-dotenv'
]

print("=" * 80)
print("📦 CHECKING AND INSTALLING DEPENDENCIES")
print("=" * 80)

missing_packages = []

# Check which packages are missing
for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"✓ {package}")
    except ImportError:
        print(f"❌ {package} (MISSING)")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Installing {len(missing_packages)} missing package(s)...")
    for package in missing_packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
            print(f"✓ Installed {package}")
        except Exception as e:
            print(f"❌ Failed to install {package}: {e}")
else:
    print("\n✓ All required packages are already installed!")

print("\n" + "=" * 80)

📦 CHECKING AND INSTALLING DEPENDENCIES
✓ langgraph
✓ langchain
✓ langchain-core
✓ langchain-openai
✓ langchain-huggingface
✓ langchain-community
❌ faiss-cpu (MISSING)
❌ python-docx (MISSING)
✓ pandas
❌ python-dotenv (MISSING)

⚠️  Installing 3 missing package(s)...
✓ Installed faiss-cpu
✓ Installed python-docx
✓ Installed python-dotenv



In [4]:
print("=" * 80)
print("✅ VERIFYING DEPENDENCY INSTALLATION")
print("=" * 80)

# Test actual imports to verify installation worked
verification_imports = {
    'langgraph': 'from langgraph.graph import StateGraph, START, END',
    'langchain_core': 'from langchain_core.documents import Document',
    'langchain_huggingface': 'from langchain_huggingface import HuggingFaceEmbeddings',
    'langchain_community': 'from langchain_community.vectorstores import FAISS',
    'docx': 'from docx import Document as DocxDocument',
    'pandas': 'import pandas as pd',
    'dotenv': 'from dotenv import load_dotenv'
}

success_count = 0
failed_imports = []

for package_name, import_statement in verification_imports.items():
    try:
        exec(import_statement)
        print(f"✓ {package_name:30} - OK")
        success_count += 1
    except Exception as e:
        print(f"❌ {package_name:30} - FAILED: {str(e)[:40]}")
        failed_imports.append(package_name)

print("\n" + "-" * 80)
if failed_imports:
    print(f"⚠️  {len(failed_imports)}/{len(verification_imports)} imports failed")
    print(f"Failed packages: {', '.join(failed_imports)}")
    print("\nAttempting to reinstall failed packages...")
    for pkg in failed_imports:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            print(f"✓ Reinstalled {pkg}")
        except:
            print(f"❌ Could not reinstall {pkg}")
else:
    print(f"✓ All {len(verification_imports)} dependencies verified successfully!")

print("=" * 80)

✅ VERIFYING DEPENDENCY INSTALLATION
✓ langgraph                      - OK
✓ langchain_core                 - OK
✓ langchain_huggingface          - OK
✓ langchain_community            - OK
✓ docx                           - OK
✓ pandas                         - OK
✓ dotenv                         - OK

--------------------------------------------------------------------------------
✓ All 7 dependencies verified successfully!


## Section 3: Load & Configure Agents

In [5]:
print("=" * 80)
print("📦 LOADING AGENT MODULES")
print("=" * 80)

# Step 1: Import AgentState first from dedicated module (no circular deps)
try:
    from app.agents.agent_state import AgentState
    print("✓ AgentState imported from agent_state")
except Exception as e:
    logger.error(f"Failed to load AgentState: {e}")
    print(f"❌ Error loading AgentState: {e}")
    raise

# Step 2: Import supervisor_node
try:
    from app.agents.supervisor_agent import supervisor_node
    print("✓ Supervisor Agent imported")
except Exception as e:
    logger.error(f"Failed to load supervisor_node: {e}")
    print(f"❌ Error loading supervisor_node: {e}")
    raise

# Step 3: Import LLMClient  
try:
    from app.llm.llm_client import LLMClient
    print("✓ LLMClient imported")
except Exception as e:
    logger.error(f"Failed to load LLMClient: {e}")
    print(f"❌ Error loading LLMClient: {e}")
    raise

# Step 4: Import RetrieverAgent (now after AgentState is available)
try:
    from app.agents.retriever_agent import RetrieverAgent
    print("✓ Retriever Agent imported")
except Exception as e:
    logger.error(f"Failed to load RetrieverAgent: {e}")
    print(f"❌ Error loading RetrieverAgent: {e}")
    raise

# Step 5: Import remaining workflow components (optional)
try:
    from app.agents.multi_agent import build_knowgen_workflow, run_knowgen_pipeline
    print("✓ Multi-Agent framework imported")
except ImportError as e:
    logger.warning(f"Workflow framework not available: {e}")
    print(f"⚠️  Workflow framework note: Complete framework not yet available (optional)")
except Exception as e:
    logger.error(f"Failed to load workflow: {e}")
    print(f"❌ Error loading workflow: {e}")

# Step 6: Initialize LLM client
try:
    llm_client = LLMClient()
    logger.info("✓ LLMClient initialized")
    print("✓ LLMClient initialized successfully")
except Exception as e:
    logger.error(f"Failed to initialize LLMClient: {e}")
    print(f"⚠️  Warning: LLMClient initialization issue - {e}")
    print("   → Check your .env file has valid API keys (OPENAI_API_KEY, etc.)")

print("\n" + "=" * 80)
print("✓ All required agents loaded and ready for testing!")
print("=" * 80)

📦 LOADING AGENT MODULES
✓ AgentState imported from agent_state


2026-04-14 11:42:20,493 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base


✓ Supervisor Agent imported
✓ LLMClient imported


2026-04-14 11:42:22,872 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-14 11:42:22,872 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base
2026-04-14 11:42:32,008 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
2026-04-14 11:42:32,011 - faiss.loader - INFO - Loading faiss with AVX2 support.
2026-04-14 11:42:32,026 - faiss.loader - INFO - Successfully loaded faiss with AVX2 support.
2026-04-14 11:42:32,026 - knowgen.retriever - INFO - FAISS index loaded successfully
2026-04-14 11:42:32,043 - __main__ - INFO - ✓ LLMClient initialized


✓ Retriever Agent imported
✓ Multi-Agent framework imported
✓ LLMClient initialized successfully

✓ All required agents loaded and ready for testing!


## Section 4: Load Sample Document

In [6]:
from docx import Document as DocxDocument
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))
from app.ingestion.extract import extract_pdf_to_md
# Check if file exists
if not os.path.exists(CNXHKH_FILE):
    print(f"⚠️  File not found: {CNXHKH_FILE}")
    print(f"\nℹ️  Update CNXHKH_FILE path in previous cell if needed")
    print(f"   Current path: {CNXHKH_FILE}")
else:
    try:
        # Load document
        doc = extract_pdf_to_md(CNXHKH_FILE)
        
        # Get file info
        file_size = os.path.getsize(CNXHKH_FILE) / (1024 * 1024)  # MB
        num_paragraphs = len(doc.paragraphs)
        num_tables = len(doc.tables)
        
        # Extract text
        full_text = "\n".join([p.text for p in doc.paragraphs])
        word_count = len(full_text.split())
        
        print("=" * 80)
        print("📄 CNXHKH.DOCX DOCUMENT INFORMATION")
        print("=" * 80)
        print(f"File: {os.path.basename(CNXHKH_FILE)}")
        print(f"Size: {file_size:.2f} MB")
        print(f"Paragraphs: {num_paragraphs}")
        print(f"Tables: {num_tables}")
        print(f"Total Characters: {len(full_text):,}")
        print(f"Total Words: {word_count:,}")
        
        print(f"\n📋 Content Preview (first 400 characters):")
        print("-" * 80)
        preview = full_text[:400]
        if len(preview) > 0:
            print(preview.strip())
            if len(full_text) > 400:
                print("...")
        else:
            print("(Empty document)")
        print("-" * 80)
        print("\n✓ Document loaded successfully!")
        
    except Exception as e:
        print(f"❌ Error loading document: {e}")
        import traceback
        traceback.print_exc()

Extracting PDF (Markdown): H:\ĐH 1,2\BT CNXHKH.docx
❌ Error loading document: 'Document' object has no attribute 'paragraphs'


Traceback (most recent call last):
  File "C:\Users\asus laptop\AppData\Local\Temp\ipykernel_25800\3292948809.py", line 18, in <module>
    num_paragraphs = len(doc.paragraphs)
                         ^^^^^^^^^^^^^^
  File "c:\Users\asus laptop\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\main.py", line 1026, in __getattr__
    raise AttributeError(f'{type(self).__name__!r} object has no attribute {item!r}')
AttributeError: 'Document' object has no attribute 'paragraphs'


## Section 5: Test Supervisor Agent

In [7]:
# Test queries in Vietnamese
test_queries = [
    "Sóng cơ học là gì?",  # What is Scientific Socialism?
    "Những đặc điểm chính của CNXH là gì?",  # What are the main characteristics of Scientific Socialism?
    "Tạo một bài quiz về chủ nghĩa Marx",  # Create a quiz about Marxism
    "So sánh CNXH và tư bản chủ nghĩa", 
    "Công thức Pythagore?" # Compare Scientific Socialism and Capitalism
]

# Store supervisor results
supervisor_results = {}

print("=" * 80)
print("🤖 TESTING SUPERVISOR AGENT")
print("=" * 80)

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}/{len(test_queries)}] {query}")
    print("-" * 80)
    
    try:
        # Create initial state
        initial_state = {"user_query": query}
        
        # Run supervisor node
        supervisor_output = supervisor_node(initial_state)
        
        # Store result
        supervisor_results[f"query_{i}"] = supervisor_output
        
        # Display results
        if supervisor_output.get('task_type'):
            print(f"✓ Task Type: {supervisor_output.get('task_type')}")
            print(f"✓ Language: {supervisor_output.get('language')}")
            print(f"✓ Intent: {supervisor_output.get('intent_summary', 'N/A')[:70]}")
            
            plan = supervisor_output.get('plan', {})
            context = plan.get('context_needed', 'N/A')
            steps = plan.get('steps', [])
            
            print(f"✓ Context Needed: {context[:70]}")
            if steps:
                print(f"✓ Steps ({len(steps)}):")
                for step_idx, step in enumerate(steps[:3], 1):
                    print(f"   {step_idx}. {step[:60]}")
        else:
            print(f"⚠️  Missing task_type in response")
            print(f"   Raw output: {supervisor_output}")
        
    except Exception as e:
        print(f"❌ Error processing query: {e}")
        import traceback
        traceback.print_exc()
        supervisor_results[f"query_{i}"] = {"error": str(e)}

print("\n" + "=" * 80)
print(f"✓ Supervisor Agent testing complete!")
print(f"   Successfully processed: {len([r for r in supervisor_results.values() if 'error' not in r])}/{len(test_queries)} queries")
print("=" * 80)

2026-04-14 11:42:42,010 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-14 11:42:42,058 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


🤖 TESTING SUPERVISOR AGENT

[Query 1/5] Sóng cơ học là gì?
--------------------------------------------------------------------------------


2026-04-14 11:42:48,126 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-14 11:42:48,136 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-14 11:42:48,136 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Task Type: qa
✓ Language: vi
✓ Intent: Người dùng muốn biết định nghĩa và giải thích về sóng cơ học.
✓ Context Needed: sóng cơ học, định nghĩa, đặc điểm, truyền sóng
✓ Steps (3):
   1. Bước 1: Sử dụng tác nhân truy xuất để tìm kiếm các tài liệu 
   2. Bước 2: Sử dụng tác nhân QA để trích xuất định nghĩa, đặc đi
   3. Bước 3: Tổng hợp thông tin để đưa ra câu trả lời đầy đủ về s

[Query 2/5] Những đặc điểm chính của CNXH là gì?
--------------------------------------------------------------------------------


2026-04-14 11:42:51,091 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-14 11:42:51,111 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-14 11:42:51,111 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


...

✓ Task Type: quiz
✓ Language: vi
✓ Intent: The user wants to generate a quiz about Marxism.
✓ Context Needed: chủ nghĩa Marx, Marxism
✓ Steps (2):
   1. Step 1: Use the retriever agent to find relevant documents a
   2. Step 2: Pass the retrieved context to the quiz generation ag

[Query 4/5] So sánh CNXH và tư bản chủ nghĩa
--------------------------------------------------------------------------------


2026-04-14 11:42:56,831 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-14 11:42:56,831 - knowgen.supervisor - INFO - === SUPERVISOR AGENT RUNNING ===
2026-04-14 11:42:56,840 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.


✓ Task Type: qa
✓ Language: vi
✓ Intent: The user wants a comparison between Socialism and Capitalism.
✓ Context Needed: Chủ nghĩa xã hội, Chủ nghĩa tư bản, CNXH, tư bản chủ nghĩa, so sánh, đ
✓ Steps (3):
   1. Step 1: Retrieve documents containing information about 'Chủ
   2. Step 2: Extract and synthesize key comparative points betwee
   3. Step 3: Generate a detailed comparison highlighting the simi

[Query 5/5] Công thức Pythagore?
--------------------------------------------------------------------------------


2026-04-14 11:42:59,640 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


✓ Task Type: qa
✓ Language: vi
✓ Intent: The user wants to know the Pythagorean formula.
✓ Context Needed: Pythagorean formula, định lý Pytago, tam giác vuông
✓ Steps (3):
   1. Step 1: Use the document retriever agent to find information
   2. Step 2: Use the QA agent to extract and present the Pythagor
   3. Step 3: Ensure the final answer is provided in Vietnamese.

✓ Supervisor Agent testing complete!
   Successfully processed: 5/5 queries


## Section 6: Test Retriever Agent

In [12]:
# 🔄 CRITICAL: Clear module cache AND re-import fresh versions
# This ensures we load the UPDATED retriever_agent.py with fixed method names

import sys
import importlib

print("=" * 80)
print("🔄 CLEARING MODULE CACHE & RELOADING MODULES")
print("=" * 80)

# List of modules to reload
modules_to_clear = [
    'app.agents.retriever_agent',
    'app.llm.llm_client',
    'app.agents.agent_state'
]

print("\n1️⃣  Removing from sys.modules cache:")
print("-" * 80)
for module_name in modules_to_clear:
    if module_name in sys.modules:
        del sys.modules[module_name]
        print(f"✓ Cleared {module_name}")
    else:
        print(f"⊘ {module_name} not in cache")

print("\n2️⃣  Re-importing fresh modules:")
print("-" * 80)

# Re-import to load fresh versions
try:
    from app.agents.agent_state import AgentState
    print("✓ Re-imported AgentState (fresh)")
except Exception as e:
    print(f"❌ Error re-importing AgentState: {e}")

try:
    from app.llm.llm_client import LLMClient
    print("✓ Re-imported LLMClient (fresh)")
except Exception as e:
    print(f"❌ Error re-importing LLMClient: {e}")

try:
    from app.agents.retriever_agent import RetrieverAgent
    print("✓ Re-imported RetrieverAgent (fresh) ← THIS NOW HAS similarity_search_with_relevance_scores()")
except Exception as e:
    print(f"❌ Error re-importing RetrieverAgent: {e}")

print("\n✓ All modules reloaded! Ready to test with updated code.")
print("=" * 80)

2026-04-14 11:52:17,625 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-14 11:52:17,628 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-14 11:52:17,629 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


🔄 CLEARING MODULE CACHE & RELOADING MODULES

1️⃣  Removing from sys.modules cache:
--------------------------------------------------------------------------------
✓ Cleared app.agents.retriever_agent
✓ Cleared app.llm.llm_client
✓ Cleared app.agents.agent_state

2️⃣  Re-importing fresh modules:
--------------------------------------------------------------------------------
✓ Re-imported AgentState (fresh)
✓ Re-imported LLMClient (fresh)


2026-04-14 11:52:23,965 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
2026-04-14 11:52:23,966 - knowgen.retriever - INFO - FAISS index loaded successfully


✓ Re-imported RetrieverAgent (fresh) ← THIS NOW HAS similarity_search_with_relevance_scores()

✓ All modules reloaded! Ready to test with updated code.


In [13]:
# Check if FAISS index exists
print("=" * 80)
print("🔍 TESTING RETRIEVER AGENT")
print("=" * 80)

retriever = None
retriever_results = {}

if not os.path.exists(VECTOR_STORE_DIR):
    print(f"\n⚠️  FAISS index not found at {VECTOR_STORE_DIR}")
    print("\n📋 HOW TO CREATE VECTOR EMBEDDINGS:")
    print("-" * 80)
    print("1. Navigate to backend directory:")
    print("   cd backend")
    print("\n2. Run ETL pipeline to process CNXHKH.docx:")
    print("   python scripts/run_etl.py")
    print("\n3. This will:")
    print("   - Extract text from CNXHKH.docx")
    print("   - Transform and chunk the content")
    print("   - Create FAISS vector embeddings")
    print("   - Save index to:", VECTOR_STORE_DIR)
    print("\n4. After ETL completes, re-run this cell")
    print("-" * 80)
    print("\n✓ Skipping Retriever Agent tests (waiting for FAISS index)")
    
else:
    # Initialize retriever
    try:
        retriever = RetrieverAgent(vector_store_dir=str(VECTOR_STORE_DIR))
        print("\n✓ Retriever Agent initialized successfully")
        
        # Test retriever with supervisor outputs
        for query_idx, (query_key, supervisor_output) in enumerate(supervisor_results.items(), 1):
            if "error" in supervisor_output:
                print(f"\n⚠️  Skipping Query {query_idx} due to supervisor error")
                continue
            
            print(f"\n{'='*80}")
            print(f"[Query {query_idx}] {test_queries[query_idx-1]}")
            print(f"{'='*80}")
            
            try:
                # Create AgentState from supervisor output
                agent_state = AgentState(
                    user_query=test_queries[query_idx-1],
                    task_type=supervisor_output.get('task_type', 'qa'),
                    language=supervisor_output.get('language', 'en'),
                    intent_summary=supervisor_output.get('intent_summary', ''),
                    plan=supervisor_output.get('plan', {})
                )
                
                # Run retriever
                retriever_output = retriever.run(agent_state)
                retriever_results[query_key] = retriever_output
                
                # Display results
                print(f"✓ Rewritten Query: {retriever_output.get('rewritten_query', 'N/A')}")
                print(f"✓ Documents Retrieved: {len(retriever_output.get('retrieved_docs', []))}")
                
                # Show top documents
                retrieved_docs = retriever_output.get('retrieved_docs', [])
                if retrieved_docs:
                    print(f"\n📄 Top Documents:")
                    for doc_idx, doc in enumerate(retrieved_docs[:3], 1):
                        score = doc.metadata.get('similarity_score', 0) if doc.metadata else 0
                        header = doc.metadata.get('header_path', 'N/A') if doc.metadata else 'N/A'
                        print(f"  [{doc_idx}] Score: {score:.3f} | Header: {header}")
                        print(f"      Preview: {doc.page_content[:80].strip()}...")
                    
                    # Show evidence
                    evidence = retriever_output.get('evidence_summary', [])
                    if evidence:
                        print(f"\n💡 Evidence Summaries ({len(evidence)} items):")
                        for ev_idx, ev in enumerate(evidence[:3], 1):
                            print(f"  {ev_idx}. {ev[:80]}")
                else:
                    print("⚠️  No documents retrieved for this query")
                
            except Exception as e:
                logger.error(f"Retriever error for query {query_idx}: {e}")
                print(f"❌ Error processing retriever: {e}")
                import traceback
                traceback.print_exc()
                retriever_results[query_key] = {"error": str(e)}
        
    except Exception as e:
        logger.error(f"Failed to initialize retriever: {e}")
        print(f"\n❌ Failed to initialize Retriever Agent: {e}")
        print(f"   Error details: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 80)
if retriever:
    print(f"✓ Retriever Agent testing complete! ({len(retriever_results)} results)")
else:
    print(f"⚠️  Retriever Agent testing skipped (FAISS index not available)")
print("=" * 80)

2026-04-14 11:52:26,682 - knowgen.retriever - INFO - Loading embedding model: intfloat/multilingual-e5-base
2026-04-14 11:52:26,690 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cpu
2026-04-14 11:52:26,690 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-base


🔍 TESTING RETRIEVER AGENT


2026-04-14 11:52:33,690 - knowgen.retriever - INFO - Loading FAISS index from: h:\KnowGen Project\backend\app\agents\vector_store\faiss_index
2026-04-14 11:52:33,690 - knowgen.retriever - INFO - FAISS index loaded successfully
2026-04-14 11:52:33,690 - google_genai.models - INFO - AFC is enabled with max remote calls: 10.



✓ Retriever Agent initialized successfully

[Query 1] Sóng cơ học là gì?


2026-04-14 11:52:42,526 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"
2026-04-14 11:52:42,526 - knowgen.retriever - INFO - Query rewritten: 'Sóng cơ học là gì?…' (18 chars) → 'Sóng cơ học định nghĩa khái niệm giải th…' (122 chars)
2026-04-14 11:52:42,526 - knowgen.retriever - INFO - Searching 20 candidates…
2026-04-14 11:52:42,626 - knowgen.retriever - INFO - Found 20 candidates
2026-04-14 11:52:42,626 - knowgen.retriever - INFO - Doc '__unknown__' pruned by relevance gate (summary_sim=0.6870 < 0.7)
2026-04-14 11:52:42,626 - knowgen.retriever - INFO - Doc-relevance gate: 0/1 doc(s) passed, 0/20 chunks retained.
2026-04-14 11:52:42,626 - knowgen.retriever - WARNING - All documents failed the doc-relevance gate (threshold=0.7). Query is off-topic.
2026-04-14 11:52:42,626 - knowgen.retriever - WARNING - No documents above confidence threshold or topic consistency filter
2026-04-14 11:52:42,626

...

✓ Rewritten Query: Quiz chủ nghĩa Marx Marxism tư tưởng lý luận triết học kinh tế chính trị Marx Engels biện chứng duy vật lịch sử giá trị thặng dư đấu tranh giai cấp bóc lột cách mạng vô sản nhà nước cộng sản hình thái kinh tế xã hội ứng dụng phê phán tranh luận thực tiễn hiện đại câu hỏi trắc nghiệm bài tập kiểm tra kiến thức hiểu biết cấu trúc dạng đáp án phân loại mức độ khó ví dụ đối lập sắc thái
✓ Documents Retrieved: 5

📄 Top Documents:
  [1] Score: 0.780 | Header: 
      Preview: passage: 2. không phát hiện ra lực lượng xã hội tiên phong có thể thực hiện cuộc...
  [2] Score: 0.777 | Header: 
      Preview: passage: đúng. chủ nghĩa xã hội khoa học có chức năng hướng dẫn giai cấp công nh...
  [3] Score: 0.774 | Header: 
      Preview: passage: câu 4: giai cấp công nhân trong cntb là giai cấp hoàn toàn không sở hữu...

💡 Evidence Summaries (5 items):
  1. passage: 2
  2. passage: đúng
  3. passage: câu 4: giai cấp công nhân trong cntb là giai cấp hoàn toàn không sở hữu

[Query 4] 

2026-04-14 11:52:54,298 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-14 11:52:54,298 - google_genai._api_client - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.84 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 5.866704243s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.d

✓ Rewritten Query: So sánh CNXH và tư bản chủ nghĩa
✓ Documents Retrieved: 5

📄 Top Documents:
  [1] Score: 0.787 | Header: 
      Preview: passage: đúng. chủ nghĩa xã hội khoa học có chức năng hướng dẫn giai cấp công nh...
  [2] Score: 0.787 | Header: 
      Preview: passage: sai. chủ nghĩa xã hội không tưởng phê phán còn không ít những hạn chế h...
  [3] Score: 0.776 | Header: 
      Preview: passage: câu 1 – tự luận: vì sao cnxh trước mác chỉ dừng lại ở một học thuyết kh...

💡 Evidence Summaries (5 items):
  1. passage: đúng
  2. passage: sai
  3. passage: câu 1 – tự luận: vì sao cnxh trước mác chỉ dừng lại ở một học thuyết kh

[Query 5] Công thức Pythagore?


2026-04-14 11:53:36,595 - httpx - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"
2026-04-14 11:53:36,600 - google_genai._api_client - INFO - Retrying google.genai._api_client.BaseApiClient._request_once in 1.79 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 23.497382925s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.

✓ Rewritten Query: Công thức Pythagore?
✓ Documents Retrieved: 0
⚠️  No documents retrieved for this query

✓ Retriever Agent testing complete! (5 results)


## Section 7: Compare & Analyze Results

In [14]:
print("=" * 80)
print("📊 COMPARATIVE ANALYSIS")
print("=" * 80)

# Aggregate statistics
analysis_results = {
    "total_queries": len(test_queries),
    "supervisor_success": 0,
    "retriever_success": 0,
    "task_type_distribution": {},
    "language_distribution": {},
    "avg_docs_retrieved": 0,
    "total_docs_retrieved": 0,
    "avg_similarity_scores": []
}

print("\n1️⃣  SUPERVISOR OUTPUT SUMMARY:")
print("-" * 80)
for query_idx, (query_key, sup_result) in enumerate(supervisor_results.items(), 1):
    if "error" not in sup_result:
        analysis_results["supervisor_success"] += 1
        task_type = sup_result.get("task_type", "unknown")
        language = sup_result.get("language", "unknown")
        
        # Update distributions
        analysis_results["task_type_distribution"][task_type] = \
            analysis_results["task_type_distribution"].get(task_type, 0) + 1
        analysis_results["language_distribution"][language] = \
            analysis_results["language_distribution"].get(language, 0) + 1
        
        intent = sup_result.get('intent_summary', 'N/A')[:50]
        print(f"[Query {query_idx}] {task_type.upper():7} | Language: {language:6} | {intent}")
    else:
        print(f"[Query {query_idx}] ERROR - {sup_result.get('error', 'Unknown error')[:50]}")

if len(retriever_results) == 0:
    print("\n2️⃣  RETRIEVER OUTPUT SUMMARY:")
    print("-" * 80)
    print("⚠️  No retriever results available")
    print("   (FAISS index may not be created yet)")
else:
    print("\n2️⃣  RETRIEVER OUTPUT SUMMARY:")
    print("-" * 80)
    for query_idx, (query_key, ret_result) in enumerate(retriever_results.items(), 1):
        if "error" not in ret_result:
            analysis_results["retriever_success"] += 1
            docs_count = len(ret_result.get("retrieved_docs", []))
            analysis_results["total_docs_retrieved"] += docs_count
            
            # Collect similarity scores
            for doc in ret_result.get("retrieved_docs", []):
                metadata = doc.metadata if hasattr(doc, 'metadata') else {}
                score = metadata.get('similarity_score', 0) if metadata else 0
                if score > 0:
                    analysis_results["avg_similarity_scores"].append(score)
            
            rewritten = ret_result.get('rewritten_query', '')[:50]
            print(f"[Query {query_idx}] Retrieved: {docs_count:2} docs | Rewritten: {rewritten}")
        else:
            print(f"[Query {query_idx}] ERROR - {ret_result.get('error', 'Unknown error')[:50]}")

if analysis_results["retriever_success"] > 0:
    analysis_results["avg_docs_retrieved"] = \
        analysis_results["total_docs_retrieved"] / analysis_results["retriever_success"]

print("\n3️⃣  STATISTICS SUMMARY:")
print("-" * 80)
print(f"✓ Total Queries: {analysis_results['total_queries']}")
print(f"✓ Supervisor Success: {analysis_results['supervisor_success']}/{analysis_results['total_queries']} ({(analysis_results['supervisor_success']/analysis_results['total_queries']*100):.0f}%)")

if analysis_results['retriever_success'] > 0 or len(retriever_results) > 0:
    retriever_rate = (analysis_results['retriever_success']/analysis_results['total_queries']*100) if analysis_results['total_queries'] > 0 else 0
    print(f"✓ Retriever Success: {analysis_results['retriever_success']}/{analysis_results['total_queries']} ({retriever_rate:.0f}%)")
    print(f"✓ Average Docs per Query: {analysis_results['avg_docs_retrieved']:.1f}")
    print(f"✓ Total Documents Retrieved: {analysis_results['total_docs_retrieved']}")
else:
    print(f"⚠️  Retriever tests not available (FAISS index not initialized)")

if analysis_results["avg_similarity_scores"]:
    avg_score = sum(analysis_results["avg_similarity_scores"]) / len(analysis_results["avg_similarity_scores"])
    min_score = min(analysis_results["avg_similarity_scores"])
    max_score = max(analysis_results["avg_similarity_scores"])
    print(f"✓ Similarity Scores: Avg={avg_score:.3f}, Min={min_score:.3f}, Max={max_score:.3f}")

if analysis_results["task_type_distribution"]:
    print(f"\n✓ Task Type Distribution: {analysis_results['task_type_distribution']}")
if analysis_results["language_distribution"]:
    print(f"✓ Language Distribution: {analysis_results['language_distribution']}")

print("\n" + "=" * 80)

📊 COMPARATIVE ANALYSIS

1️⃣  SUPERVISOR OUTPUT SUMMARY:
--------------------------------------------------------------------------------
[Query 1] QA      | Language: vi     | Người dùng muốn biết định nghĩa và giải thích về s
[Query 2] QA      | Language: vi     | The user wants to know the main characteristics of
[Query 3] QUIZ    | Language: vi     | The user wants to generate a quiz about Marxism.
[Query 4] QA      | Language: vi     | The user wants a comparison between Socialism and 
[Query 5] QA      | Language: vi     | The user wants to know the Pythagorean formula.

2️⃣  RETRIEVER OUTPUT SUMMARY:
--------------------------------------------------------------------------------
[Query 1] Retrieved:  0 docs | Rewritten: Sóng cơ học định nghĩa khái niệm giải thích bản ch
[Query 2] Retrieved:  5 docs | Rewritten: Chủ nghĩa Xã hội khoa học đặc điểm chính tính chất
[Query 3] Retrieved:  5 docs | Rewritten: Quiz chủ nghĩa Marx Marxism tư tưởng lý luận triết
[Query 4] Retrieved:  5 do